[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/doubled_algebra/doubled_algebra.ipynb)

# The Doubled Algebra

This example builds a larger algebra in which every linear map on the multivectors of a space is itself a multivector. Giving each direction of the space a partner of the opposite square doubles the algebra, and in the doubled algebra adding a direction to a multivector, removing one, and every combination of the two are products: a map is an element, and applying maps one after another is multiplying them.

Each direction `x` and its partner `u` make two null vectors, `(x + u) / 2`, which adds `x` to a blade as a wedge does, and `(x - u) / 2`, which removes it as a contraction does. The original multivectors sit in a left ideal of the doubled algebra, so left multiplication by any element is a map on them, `State <- State`, and with n directions the doubled algebra has exactly as many components, 4ⁿ, as there are such maps: each map is one multivector, recovered by a trace pairing and one solve, and the composition of maps is the geometric product of their multivectors. A product of an adding and a removing vector is a dyad that acts on every factor of a blade, so its exponential carries areas and volumes along with vectors.

In the notation of fermionic Fock space the adding and removing vectors read as creation and annihilation operators $a_i^\dagger$ and $a_i$, with $\{a_i, a_j^\dagger\} = \delta_{ij}$, and the statement as $\mathrm{Cl}(n, n) \cong \mathrm{End}(\Lambda \mathbb R^n)$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
from functools import reduce
from operator import mul

import numpy as np

from numga import Algebra, NumpyContext, stack
from examples.util.animation import save_figure
from examples.math.doubled_algebra import render
from examples.math.doubled_algebra.core import grid_samples

np.set_printoptions(precision=3, suppress=True)

In [ ]:
# Two positive and two negative directions give Cl(2,2), with sixteen blades.
ga = Algebra("x+y+u-v-")
context = NumpyContext(ga)
mv = context.multivector
Full = ga.gatype.full()
# The states have the four components of a planar multivector: 1, x, y, xy.
State = Full.derive.within("x y")
Vector = ga.gatype.vector().derive.within("x y")
StateMap = ga.gatype((State, State))
state_components = len(State.output_subspace)

## 1. Adding and removing a direction

Wedging a direction into a blade adds that direction, while contraction removes it, with the sign set by its position in the blade. Each operation squares to zero. Pairing a positive and a negative direction produces two null vectors whose products have exactly those properties. Their mixed anticommutator is the pairing between a direction and its measurement.

In [ ]:
directions = stack([mv.x, mv.y])
partners = stack([mv.u, mv.v])
creators = (directions + partners) / 2
removers = (directions - partners) / 2

# Each family consists of null vectors. The mixed pair has a nonzero pairing.
create_x, create_y = creators
remove_x, remove_y = removers
pairing_x = 2 * remove_x.anticommutator(create_x)

print("Add x")
print(create_x.coefficient_table())
print("Remove x")
print(remove_x.coefficient_table())
print("Their mixed anticommutator")
print(pairing_x.coefficient_table())

## 2. Multiplication as a map on states

An empty state is killed by either remover. Multiplying it by creators supplies the other states, all within a four-dimensional left ideal: a space closed under multiplication from the left. The projector embeds the four independent coefficients into this ideal; reading them out undoes that embedding. Multiplying between these two steps gives a unary extensor, `State <- State`.

In [ ]:
# remove_x * create_x keeps states without x; the two such projectors commute.
projector = reduce(mul, removers * creators)
embedder = State * projector                                      # [] Full <- State
# Keep the independent components. Each projector contributes a factor of one half.
state_readout = (state_components * Full).cast(State)              # [] State <- Full
representations = state_readout(Full * embedder)                   # [] State <- (Full, State)

# Binding the multiplying element leaves its state argument open.
insertion = representations(create_x)                            # [] State <- State
removal = representations(remove_x)                               # [] State <- State

# The same maps can be built directly on the planar states.
wedge = mv.x ^ State
contraction = mv.x.left_contraction(State)

print("Add x: State <- State")
print(insertion.coefficient_table())
print("Remove x: State <- State")
print(removal.coefficient_table())

In [ ]:
state = 1 + 2 * mv.x - mv.y + 3 * mv.xy

# create_x + remove_x is mv.x: ordinary geometric multiplication adds and removes.
familiar_product = representations(create_x + remove_x)         # [] State <- State
result = familiar_product(state)                                 # [] State

print("State")
print(state.coefficient_table())
print("Multiplication by x: State <- State")
print(familiar_product.coefficient_table())
print("Result")
print(result.coefficient_table())

## 3. A shear from one mixed product

Removing `y` and then adding `x` sends `y` to `x` and kills `x`: a dyad. On a blade it replaces each factor in turn, which is the product rule defining a derivation. Exponentiating the same element therefore shears the vectors and carries their wedge with them. The square becomes a parallelogram while its area stays fixed.

In [ ]:
amount = 0.8
extent, lines = 1.5, 9
boundary = mv(Vector, [[-1, -1], [1, -1], [1, 1], [-1, 1]])
grid = grid_samples(directions, extent, lines)

# The right factor acts first: remove y, then add x.
generator = create_x * remove_y
# This generator squares to zero, so its exponential terminates after the linear term.
transformation = representations((generator * amount).exp())      # [] State <- State
moved_grid = transformation(grid)
moved_boundary = transformation(boundary)
moved_directions = transformation(directions)
carried_area = transformation(mv.xy)

print("Generator: State <- State")
print(representations(generator).coefficient_table())
print("Carried area")
print(carried_area.coefficient_table())
save_figure(render.draw_shear(grid, moved_grid, boundary, moved_boundary, moved_directions), "doubled_algebra_shear");

The grey grid and dashed square show the reference shape; the blue grid and orange boundary show the shear. The arrows are the images of `x` and `y`: the horizontal direction stays fixed while the other tilts towards it.

## 4. Products beyond vector maps

Removing and reinserting a direction keeps precisely the blades containing it. Summing over directions therefore counts a blade's grade. Products of two creators or two removers do something different: they add or remove a pair of directions. The exchange below swaps scalar and area components while killing both vector components, so its action cannot be inferred from its restriction to vectors.

In [ ]:
# Each term counts whether its direction occurs in the state.
count = (creators * removers).sum(axis=0)
# Add both directions, or remove them in the reverse order.
exchange = create_x * create_y + remove_y * remove_x
exchanged = representations(exchange)(state)

print("Count directions: State <- State")
print(representations(count).coefficient_table())
print("Exchange scalar and area: State <- State")
print(representations(exchange).coefficient_table())
print("Exchanged state")
print(exchanged.coefficient_table())

## 5. Every unary map as a multivector

The sixteen independent elements of this algebra represent all maps on four state components. A map can be read back into a multivector by tracing it against the open representation and solving the scalar-product pairing. The two printed maps below agree; the multivector between them contains the same information, and composing maps corresponds to multiplying their encoded multivectors.

In [ ]:
measured = 2 * mv.x + mv.y
returned = 1 + mv.x - mv.xy
coupling = 0.3

# A scalar measurement with a mixed-grade return, plus insertion of y.
state_map = returned * measured.scalar_product(State) + coupling * (mv.y ^ State)
# Trace over the State output and input; the algebra argument remains open.
pairing = Full.scalar_product(Full)
multivector = pairing.solve(representations(Full, state_map).trace(0, 2) / state_components)
recovered = representations(multivector)                          # [] State <- State

print("Prescribed map: State <- State")
print(state_map.coefficient_table())
print("Encoded multivector")
print(multivector.coefficient_table())
print("Recovered map: State <- State")
print(recovered.coefficient_table())

## 6. Four directions, eight generators

Four directions and their four partners give Cl(4,4): 256 components acting on the 16 components of a four-dimensional multivector. The construction is unchanged. A single mixed product still has only four blade coefficients; acting on `yzt` replaces `y` with `x`, without touching the other factors.

In [ ]:
ga4 = Algebra("x+y+z+t+u-v-w-s-")
context4 = NumpyContext(ga4)
mv4 = context4.multivector
Full4 = ga4.gatype.full()
State4 = Full4.derive.within("x y z t")
state_components4 = len(State4.output_subspace)

In [ ]:
directions4 = stack([mv4.x, mv4.y, mv4.z, mv4.t])
partners4 = stack([mv4.u, mv4.v, mv4.w, mv4.s])
creators4 = (directions4 + partners4) / 2
removers4 = (directions4 - partners4) / 2

projector4 = reduce(mul, removers4 * creators4)
embedder4 = State4 * projector4
state_readout4 = (state_components4 * Full4).cast(State4)
representations4 = state_readout4(Full4 * embedder4)

# The same dyad acts on every factor of a higher-grade blade.
create_x4 = (mv4.x + mv4.u) / 2
remove_y4 = (mv4.y - mv4.v) / 2
generator4 = create_x4 * remove_y4
state4 = mv4.yzt
result4 = representations4(generator4)(state4)

print("Generator")
print(generator4.coefficient_table())
print("State")
print(state4.coefficient_table())
print("Result")
print(result4.coefficient_table())

In [ ]:
# checks
# Adding and removing x are the wedge and the contraction; the shear keeps the area; the encoded
# multivector gives back the prescribed map; and composing two maps is multiplying their multivectors.
np.testing.assert_allclose((insertion - wedge).kernel, 0, atol=1e-12)
np.testing.assert_allclose((removal - contraction).kernel, 0, atol=1e-12)
np.testing.assert_allclose((carried_area - mv.xy).kernel, 0, atol=1e-12)
np.testing.assert_allclose((recovered - state_map).kernel, 0, atol=1e-12)
composed = representations(create_x * remove_y)
np.testing.assert_allclose((composed - insertion(representations(remove_y))).kernel, 0, atol=1e-12)

This wedge-and-contraction representation is the spinor construction for a split algebra; the exterior states themselves are its spinors. Gualtieri develops it in [Generalized complex geometry, §1.3](https://www.math.toronto.edu/mgualt/gcg/gcg.pdf#page=5), alongside the transformations that mix the two null families.